In [2]:
#1.1 Finding where the leaks originate from
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from statsmodels.stats.proportion import proportion_confint

# ---------- 1. LOAD AND DEFINE VARIABLES ----------
df = pd.read_excel("Competition_Dataset_Final.xlsx")
df["leak"] = (df["Failure Category"] == "Leak").astype(int)       # outcome
df["F2"]   = (df["Facility"] == "Facility 2").astype(int)          # predictor 1
df["SB"]   = (df["Coil Supplier"] == "Supplier B").astype(int)     # predictor 2
df["F2xSB"]= df["F2"] * df["SB"]

# ---------- 2. THE 2x2 TABLE WITH EXACT INTERVALS ----------
cells = df.groupby(["F2", "SB"])["leak"].agg(k="sum", n="count")
cells["rate"] = cells.k / cells.n
ci = [proportion_confint(k, n, method="beta") for k, n in zip(cells.k, cells.n)]  # Clopper-Pearson: safe at 0
cells["lo"], cells["hi"] = [c[0] for c in ci], [c[1] for c in ci]
print(cells.round(3), "\n")

# ---------- 3. CANDIDATE MODELS (each: fit on train -> predict on test) ----------
EPS = 1e-3
def fit_null(tr, te):
    return np.full(len(te), tr.leak.mean())

def _logit(cols):
    def f(tr, te):
        m = LogisticRegression(C=1.0, max_iter=1000).fit(tr[cols], tr.leak)   # L2 penalty handles separation
        return m.predict_proba(te[cols])[:, 1]
    return f
fit_additive    = _logit(["F2", "SB"])
fit_interaction = _logit(["F2", "SB", "F2xSB"])

def fit_either(tr, te):   # one switch: risk = F2 OR SB; rate per group (lightly smoothed)
    r = ((tr.F2 | tr.SB) == 1)
    p1 = (tr.leak[r].sum() + .5) / (r.sum() + 1); p0 = (tr.leak[~r].sum() + .5) / ((~r).sum() + 1)
    return np.where((te.F2 | te.SB) == 1, p1, p0)

def _or_pred(theta, F2, SB):
    a, b, c = 1 / (1 + np.exp(-theta))           # base, F2 effect, SB effect (all in 0-1)
    return 1 - (1 - a) * (1 - b * F2) * (1 - c * SB)
def fit_noisyor(tr, te):
    def nll(t):
        p = np.clip(_or_pred(t, tr.F2.values, tr.SB.values), 1e-9, 1 - 1e-9)
        return -(tr.leak.values * np.log(p) + (1 - tr.leak.values) * np.log(1 - p)).sum()
    th = minimize(nll, [-3, 0, 0], method="Nelder-Mead", options={"maxiter": 4000}).x
    return _or_pred(th, te.F2.values, te.SB.values)

models = {"null": fit_null, "additive": fit_additive, "noisy-OR": fit_noisyor,
          "either-factor switch": fit_either, "interaction": fit_interaction}

# ---------- 4. REPEATED CROSS-VALIDATION ----------
def logloss(y, p):
    p = np.clip(p, EPS, 1 - EPS); return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
strat = df.F2.astype(str) + df.SB.astype(str)          # stratify on the 4 cells so every fold has all of them
scores = {k: [] for k in models}
for tr_i, te_i in cv.split(df, strat):
    tr, te = df.iloc[tr_i], df.iloc[te_i]
    for name, fn in models.items():
        scores[name].append(logloss(te.leak.values, fn(tr, te)))
print(pd.DataFrame({k: [np.mean(v), np.std(v)] for k, v in scores.items()}, index=["mean", "sd"]).T.round(4), "\n")

# ---------- 5. ARE THE TWO FACTORS INDEPENDENT? (bootstrap test on the 'both' cell) ----------
rng = np.random.default_rng(0)
def or_both(d):
    a = d[(d.F2 == 1) & (d.SB == 0)].leak.mean(); b = d[(d.F2 == 0) & (d.SB == 1)].leak.mean()
    return 1 - (1 - a) * (1 - b)          # what independence would predict for units with both
gaps = []
for _ in range(2000):
    s = df.sample(len(df), replace=True, random_state=int(rng.integers(1e9)))
    gaps.append(s[(s.F2 == 1) & (s.SB == 1)].leak.mean() - or_both(s))
obs = df[(df.F2 == 1) & (df.SB == 1)].leak.mean()
print(f"observed 'both' rate {obs:.3f} | independence predicts {or_both(df):.3f} | "
      f"gap 95% CI {np.percentile(gaps, [2.5, 97.5]).round(3)}\n")

# ---------- 6. DOES ANYTHING ELSE MATTER INSIDE THE RISK GROUP? ----------
risk = df[(df.F2 | df.SB) == 1]
for v in ["Shift", "Product Family", "Customer Region", "Controls Revision", "Facility", "Coil Supplier"]:
    t = pd.crosstab(risk[v], risk.leak); print(f"{v:18s} chi2 p = {chi2_contingency(t)[1]:.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'Competition_Dataset_Final.xlsx'